# Transform SRS and get Vertices of the Geometry

This example shows how to transform the spatial reference system of a polygon and extract its vertices using the coordinates of the new spatial reference system. However, this can lead to problems when the geometry intersects the antimeridian (±180° longitude line) in the original SRS. How to resolve this issue is demonstrated later. 

## Transform Spatial Reference System and Extract Vertices

First, create a polygon, then extract its vertices.

In [ ]:
import geokit.core.srs
import geokit.core.geom
import matplotlib.pyplot as plt

In [ ]:
aachen_pt = geokit.core.geom.point((6.083, 52.775), srs=geokit.core.srs.EPSG4326)
print(type(aachen_pt))
# Get the buffer around this point
aachen_buffered_area = aachen_pt.Buffer(1)
# Get a boundary
boundary_polygon = aachen_buffered_area.Boundary()
geokit.core.geom.extractVerticies(boundary_polygon)[:5, :]

### Transform Polygon and Extract Vertrices

The polygon has been converted from SRS EPSG4326 to EPSG3857, and its vertices have been extracted.

In [ ]:
# Get Points
boundary_polygon_3857 = geokit.core.geom.transform(boundary_polygon, toSRS=geokit.core.srs.EPSG3857)
geokit.core.geom.extractVerticies(boundary_polygon_3857)[:5, :]

### Compare the transformed Geometry 

Lastly, the original geometry and the transformed geometry are compared. As expected, the original shape is distorted due to the spatial transformation.

In [ ]:
fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(12, 6))
ax_handle = geokit.core.geom.drawGeoms(boundary_polygon, figsize=(6, 6), ax=axs[0])
ax_handle = geokit.core.geom.drawGeoms(boundary_polygon_3857, figsize=(6, 6), ax=axs[1])

## Deal with antimeridian issues

The ```transform()``` method shown hereabove can lead to problems when the geometry in its original SRS would intersect with the antimeridian (+/-180° longitude line).
Geokit contains several functions to deal with such issues.

1. Apply the transform() function with revert360degProj flag to preserve geometry shape at the antimeridian
2. Force the resulting geometry into the -/+180° range window
3. Split given multipolygons which have been adapted to the -/+180° range but would actually be connected across the antimeridian

In [ ]:
import pathlib
import geokit.core.geom
import geokit.core.vector
import geokit.core.srs

In [ ]:
# Apply to a geometry close to the antimeridian
# NOTE: The inbuilt PROJ function would distort or even break the geometry since it shifts points "behind the antimeridian" by 360° - when reconnecting those with the non-shifted points, the geometry is broken.

# create a polygon near the antimeridian
polygon_4326 = geokit.core.geom.box((-179.9, -1, -175.9, 1), srs=geokit.core.srs.EPSG4326)
# now transform it to a centered LAEA projection
_laea = geokit.core.srs.centeredLAEA(geom=polygon_4326)
polygon_laea = geokit.core.geom.transform(polygon_4326, toSRS=_laea)

# buffer the polygon and plot
buffered_polygon_laea = polygon_laea.Buffer(100000)
geokit.core.geom.drawGeoms(buffered_polygon_laea, srs=_laea, figsize=(8, 4))

# now try to reproject to EPSG:4326 the "standard" way
buffered_polygon_4326_broken = geokit.core.geom.transform(buffered_polygon_laea, toSRS=4326)
print(buffered_polygon_4326_broken.GetEnvelope())
geokit.core.geom.drawGeoms(buffered_polygon_4326_broken, srs=4326, figsize=(8, 4))
# NOTE: Self-intersection and a polygon that ranges nearly across the whole world! See plot but also envelope above

# now use the revert360degProj = True flag to fix the shifted points
buffered_polygon_4326_fixed = geokit.core.geom.transform(buffered_polygon_laea, toSRS=4326, revert360degProj=True)
print(buffered_polygon_4326_fixed.GetEnvelope())
geokit.core.geom.drawGeoms(buffered_polygon_4326_fixed, srs=4326, figsize=(8, 4))
# NOTE how the polygon shape is correct now, with an extent over the antimeridian though!

In [ ]:
# what if we need the polygon to be in the -180°/+180° longitude range convention?
# use the fixOutOfBoundsGeom() method to split and shift or clip the geometry

# first option: clip off whatever is beyond the -180°/180° range
buffered_polygon_4326_fixed_clipped = geokit.core.geom.fixOutOfBoundsGeoms(buffered_polygon_4326_fixed, how="clip")
geokit.core.geom.drawGeoms(buffered_polygon_4326_fixed_clipped, figsize=(3, 3))
# NOTE: Only the part in the -180°/180° range is kept (here the part on the left of the antimeridian)

# second option: shift clipped parts beyond the -180°/180° range back into it
buffered_polygon_4326_fixed_shifted = geokit.core.geom.fixOutOfBoundsGeoms(buffered_polygon_4326_fixed, how="shift")
geokit.core.geom.drawGeoms(buffered_polygon_4326_fixed_shifted)
# NOTE: The polygon is now in the -180°/180° range, but it is split into two parts

In [ ]:
# Now we can also have multipolygons which were provided in a split version to observe the -/+180° latitude range convention
# Think of Fidji, a country spanning the antimeridian

# load the shape file of Fidji, a country spanning the antimeridian
import pathlib

from geokit.core.get_test_data import get_test_data

data_cache_folder = pathlib.Path().cwd().parent.parent.parent.joinpath("geokit", "data")

FJI_geom = geokit.core.vector.extractFeatures(
    get_test_data(
        file_name="FJI.shp",
        data_cache_folder=data_cache_folder,
    )
).geom.iloc[0]

# FJI is split into two parts, one in the eastern hemisphere and one in the western hemisphere separated by the antimeridian
# in our example like mostly, the Western part has been shifted eastwards by 360 degrees
# that creates a problem when plotting the data or loading anything via the extent due to its huge extent of 360° longitude
# it also prevents distance measuring between parts east and west of the antimeridian

# the test plot demonstrates the problem
geokit.core.geom.drawGeoms(FJI_geom)
# NOTE how far it spreads, with some islands on the right and some on the left of the plot

# we can therefore extract each of the two parts separately
# demonstrate here for the part "right" of the antimeridian (i.e. East of antimeridian but West on the map)
FJI_right_geom = geokit.core.geom.divideMultipolygonIntoEasternAndWesternPart(geom=FJI_geom, side="right")
geokit.core.geom.drawGeoms(FJI_right_geom, figsize=(4, 4))

# (side='left' would return the part West of the antimeridian, i.e. East on the map)
# (side='both' would return both parts as a tuple of 2 geometries)

# side='main' returns the side with the largest total polygon area
FJI_main_geom = geokit.core.geom.divideMultipolygonIntoEasternAndWesternPart(geom=FJI_geom, side="main")
geokit.core.geom.drawGeoms(FJI_main_geom, figsize=(4, 4))
# NOTE: we can see that in our case, it is the part "left" of the antimeridian that has the largest area